# CSDI: model test

Loading the model from `https://github.com/ermongroup/CSDI.git` and usign the pre-trained model to test with my time series and validate feasibility

In [1]:
from pathlib import Path
import sys
import pandas as pd
from torch.utils.data import DataLoader, Dataset, Subset
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from datetime import datetime

In [2]:
ticker = "AAPL"
start_date = "2010-01-01"
end_date = "2024-12-31"

# training parameters
sequence_length = 252
validation_ratio = 0.2

execute_train = True
checkpoint_path = "../checkpoints/checkpoint_epoch_40.pth"
num_samples = 100
date_to_load = "2026-01-08"

In [3]:
# 1) go back to parent folder
p1 = Path.cwd().parent

# 2) go back to parent of the parent
p2 = Path.cwd().parent.parent

# 3) enter folder "CSDI" (assumed to be inside that parent-of-parent)
csdi_dir = p2 / "CSDI"

# Make sure it exists (optional but recommended)
if not csdi_dir.is_dir():
    raise FileNotFoundError(f"Folder not found: {csdi_dir}")

# Add CSDI to Python import path so imports work from anywhere
sys.path.insert(0, str(csdi_dir))

dataset_dir = p1 / "data"
config_dir = p1 / "configs"

sys.path.insert(0, str(dataset_dir))

In [4]:
# 4) imports
import torch
from main_model import CSDI_base
from yahoo_data import get_dataloader, FinancialDataset


In [5]:
import yaml

def load_config(config_path):
    """
    Reads a YAML configuration file and returns a nested dictionary.
    
    Args:
        config_path (str): Path to the .yaml file.
        
    Returns:
        dict: Configuration parameters for CSDI initialization.
    """
    try:
        with open(config_path, "r") as f:
            # safe_load handles standard YAML tags and avoids code injection
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        raise FileNotFoundError(f"Config file not found at {config_path}")
    except yaml.YAMLError as exc:
        raise Exception(f"Error parsing YAML file: {exc}")

# Usage for your thesis project
config = load_config("../configs/base_csdi.yaml")

# Accessing parameters (matches the CSDI_base logic)
target_dim = config["model"]["target_dim"]
beta_start = config["diffusion"]["beta_start"]

print(f"Initialized with target_dim: {target_dim}")

Initialized with target_dim: 5


In [6]:
class CSDI_Financial(CSDI_base):
    def __init__(self, config, device, target_dim=5):
        super().__init__(target_dim, config, device)

    def process_data(self, batch):
        # Maps your FinancialDataset dictionary to the model

        observed_data = batch["observed_data"].to(self.device).float()
        observed_mask = batch["observed_mask"].to(self.device).float()
        observed_tp = batch["timepoints"].to(self.device).float()
        gt_mask = batch["gt_mask"].to(self.device).float()

        cond_mask = batch.get("cond_mask", observed_mask).to(self.device).float()

        cut_length = torch.zeros(len(observed_data)).long().to(self.device)

        # Ensure (B, K, L) shape
        return (observed_data, observed_mask, observed_tp, gt_mask, cond_mask, cut_length)

In [7]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = pd.read_csv(f'../data/{ticker}_{start_date}_{end_date}_processed.csv')
print(df)

def create_dataloaders(df, seq_len, val_rat, batch_size):
    # 1. Create full dataset
    full_dataset = FinancialDataset(dataset=df, seq_len=seq_len, close_idx=3, forecast_horizon=None)
    print("Full dataset Okay")
    
    # 2. Calculate split point (Chronological)
    total_len = len(full_dataset)
    val_len = int(total_len * val_rat)
    train_len = total_len - val_len
    
    # 3. Create subsets
    # Training gets the earlier data, Validation gets the most recent
    train_dataset = Subset(full_dataset, range(0, train_len))
    print("Train Okay")
    val_dataset = Subset(full_dataset, range(train_len, total_len))
    print("Validation Okay")
    
    # 4. Create Loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    print("Train Loader Okay")
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    print("Validation Loader Okay")
    return train_loader, val_loader

          Open      High       Low     Close    Volume
0     0.183717 -0.232828  0.379857  0.043035  0.635150
1    -0.053535 -0.486570 -0.497403 -0.968573 -0.273358
2     0.264426 -0.418403  0.017449 -0.160763 -0.464357
3    -0.168128 -0.280224  0.140662  0.322050 -0.201814
4     0.279136 -0.423615 -0.478908 -0.560005  0.104901
...        ...       ...       ...       ...       ...
3767  0.041223 -0.442469  0.344968  0.118928 -4.098826
3768  0.020710  0.072896  0.615755  0.594633 -1.801145
3769 -0.056859 -0.234643  0.467274  0.125221  0.509984
3770 -0.450278 -0.874233 -0.899743 -0.814587  1.412766
3771 -1.193710 -1.394892 -0.629791 -0.815810 -0.556674

[3772 rows x 5 columns]


In [8]:
train_loader, val_loader = create_dataloaders(df, sequence_length, validation_ratio, batch_size=config['train']['batch_size'])

Full dataset Okay
Train Okay
Validation Okay
Train Loader Okay
Validation Loader Okay


# Core statistics

In [9]:
print("Descriptive Statistics for Standardized ticker: ", ticker)
print(df.describe())

Descriptive Statistics for Standardized ticker:  AAPL
               Open          High           Low         Close        Volume
count  3.772000e+03  3.772000e+03  3.772000e+03  3.772000e+03  3.772000e+03
mean  -8.928076e-11 -1.447193e-08 -1.117504e-08  8.205493e-09  8.020541e-09
std    1.000132e+00  1.000132e+00  1.000132e+00  1.000132e+00  1.000133e+00
min   -1.201118e+01 -8.145225e+00 -1.564787e+01 -7.898292e+00 -5.660525e+00
25%   -3.607679e-01 -5.361357e-01 -3.761705e-01 -4.784359e-01 -6.256660e-01
50%    1.616051e-02 -1.359673e-01  1.656052e-01  1.827763e-03 -5.402130e-02
75%    3.940182e-01  4.203780e-01  5.450312e-01  5.314068e-01  5.893394e-01
max    8.065092e+00  8.187183e+00  5.700387e+00  6.389341e+00  5.774343e+00


In [10]:
# features = df.columns
# num_features = len(features)

# # Use a single figure with subplots to minimize overhead
# fig, axes = plt.subplots(num_features, 1, figsize=(8, 4 * num_features))

# for i, feature in enumerate(features):
#     ax = axes[i]
#     ax.hist(df[feature].dropna(), bins=50, alpha=0.7, color='blue')
#     ax.set_title(f"Histogram of {feature} of {ticker}")
#     ax.set_xlabel(feature)
#     ax.set_ylabel("Frequency")

# plt.tight_layout()
# plt.show()

# # CRITICAL: Close the figure object to free memory
# plt.close(fig)

# Training from Scratch

In [11]:
# Initialize your custom model and move to device
model = CSDI_Financial(config, device).to(device)

# Standard optimizer for CSDI
optimizer = optim.Adam(model.parameters(), lr=config["train"]["lr"])

c:\Users\Lenovo\anaconda3\envs\ts_diffusion\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


# Feasibility Check: Training

In [12]:
if execute_train == True:
    today_str = datetime.now().strftime("%Y-%m-%d")
    history = {"train_loss": [], "val_loss": [], "volatility_error": []}

    model.train()
    for epoch in range(config["train"]["epochs"]):
        cumulative_loss = 0

        progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}")

        for i, batch in enumerate(train_loader):
            optimizer.zero_grad()
            
            # The forward call calculates the MSE between added noise and predicted noise
            loss = model(batch, is_train=1) 
            
            loss.backward()
            optimizer.step()
            
            cumulative_loss += loss.item()
            progress_bar.set_postfix({"loss": loss.item()})
            
            # Stop if we hit the limit defined in your config
            if i >= config["train"]["itr_per_epoch"]:
                break
                
        avg_loss = cumulative_loss / (i + 1)
        history['train_loss'].append(avg_loss)
        # print(f"Epoch {epoch}: Avg Loss = {avg_loss:.6f}")

        with torch.no_grad():
            # Compute Validation Loss on a hold-out batch
            val_batch = next(iter(val_loader)) # Assume you created a val_loader
            val_loss = model(val_batch, is_train=0) # CSDI_base uses is_train=0 for validation [cite: 3156]
            history["val_loss"].append(val_loss.item())

            # 3. Core Financial Statistic: Volatility Alignment
            # Generate a small sample to check if the model overshoots variance [cite: 5869]
            samples, observed, target_mask, _, _ = model.evaluate(val_batch, n_samples=5)
            # Flatten samples to compute global volatility of generated log-returns
            gen_vol = samples.std().item()
            real_vol = observed.std().item()
            vol_err = abs(gen_vol - real_vol)
            history["volatility_error"].append(vol_err)

        print(f"\n[Epoch {epoch}] Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f} | Vol Error: {vol_err:.4f}")
        model.train()
        
        # Save checkpoint periodically
        if epoch % 10 == 0:
            torch.save(model.state_dict(), f"../checkpoints/checkpoint_epoch_{epoch}_{today_str}.pth")

    torch.save(model.state_dict(), f"../checkpoints/checkpoint_epoch_{epoch}_{today_str}.pth")

Epoch 0:   0%|          | 0/353 [04:27<?, ?it/s, loss=0.432] 


[Epoch 0] Train Loss: 0.4990 | Val Loss: 0.0000 | Vol Error: 227.0985


Epoch 0:   0%|          | 0/353 [14:08<?, ?it/s, loss=0.432]



[Epoch 1] Train Loss: 0.3483 | Val Loss: 0.0000 | Vol Error: 224.3222


Epoch 2:   0%|          | 0/353 [04:29<?, ?it/s, loss=0.102] 

...


[Epoch 3] Train Loss: 0.3226 | Val Loss: 0.0000 | Vol Error: 214.0720


Epoch 4:   0%|          | 0/353 [04:20<?, ?it/s, loss=0.413] 


[Epoch 4] Train Loss: 0.3510 | Val Loss: 0.0000 | Vol Error: 213.8199


Epoch 4:   0%|          | 0/353 [14:05<?, ?it/s, loss=0.413]


KeyboardInterrupt: 

In [ ]:
if execute_train == True:
    import json

    # Define path for history
    history_path = f"../checkpoints/{ticker}_{today_str}_training_history.json"

    # Save dictionary to file
    with open(history_path, 'w') as f:
        json.dump(history, f)

    print(f"History saved to {history_path}")

## Plotting and summarising info

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def summarize_series(name, values):
    arr = np.asarray(values, dtype=float)
    out = {
        "name": name,
        "n": int(arr.size),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr, ddof=1)) if arr.size > 1 else float("nan"),
        "last": float(arr[-1]) if arr.size else float("nan"),
    }
    # simple trend: slope of linear fit vs epoch index
    if arr.size > 1:
        x = np.arange(arr.size)
        out["slope"] = float(np.polyfit(x, arr, 1)[0])
    else:
        out["slope"] = float("nan")
    return out

def print_summary_table(history):
    keys = ["train_loss", "val_loss", "volatility_error"]
    stats = [summarize_series(k, history.get(k, [])) for k in keys]

    header = f"{'metric':<18}{'n':>6}{'min':>12}{'max':>12}{'mean':>12}{'std':>12}{'last':>12}{'slope/ep':>12}"
    print(header)
    print("-" * len(header))
    for s in stats:
        print(
            f"{s['name']:<18}{s['n']:>6}"
            f"{s['min']:>12.6g}{s['max']:>12.6g}{s['mean']:>12.6g}"
            f"{s['std']:>12.6g}{s['last']:>12.6g}{s['slope']:>12.6g}"
        )

def plot_history(history, smooth_window=5):
    epochs = np.arange(len(history["train_loss"]))

    def smooth(y, w):
        y = np.asarray(y, dtype=float)
        if w is None or w <= 1 or y.size < w:
            return y
        kernel = np.ones(w) / w
        return np.convolve(y, kernel, mode="valid")

    # -------- Loss curves (train & val) --------
    plt.figure()
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Train/Val Loss per Epoch")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # -------- Smoothed loss (optional) --------
    if smooth_window and smooth_window > 1:
        tr = np.asarray(history["train_loss"], dtype=float)
        va = np.asarray(history["val_loss"], dtype=float)

        tr_s = smooth(tr, smooth_window)
        va_s = smooth(va, smooth_window)

        ep_s = np.arange(len(tr_s)) + (smooth_window - 1)  # align "valid" convolution

        plt.figure()
        plt.plot(epochs, tr, alpha=0.3, label="train_loss (raw)")
        plt.plot(ep_s, tr_s, label=f"train_loss (MA{smooth_window})")
        plt.plot(epochs, va, alpha=0.3, label="val_loss (raw)")
        plt.plot(ep_s, va_s, label=f"val_loss (MA{smooth_window})")
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.title("Smoothed Loss (Moving Average)")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    # -------- Volatility error curve --------
    plt.figure()
    plt.plot(epochs, history["volatility_error"], label="volatility_error")
    plt.xlabel("epoch")
    plt.ylabel("abs(std(gen) - std(real))")
    plt.title("Volatility Alignment Error per Epoch")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# ---- Usage (after training finishes) ----
print_summary_table(history)
plot_history(history, smooth_window=5)


In [ ]:
def plot_training_results(history):
    epochs = range(len(history['train_loss']))
    
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # 1. Plot Loss (Training and Validation)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss (MSE)', color='tab:blue')
    ax1.plot(epochs, history['train_loss'], label='Train Loss', color='tab:blue', linestyle='--')
    ax1.plot(epochs, history['val_loss'], label='Val Loss', color='tab:cyan')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax1.legend(loc='upper left')

    # 2. Create secondary axis for Volatility Error
    ax2 = ax1.twinx() 
    ax2.set_ylabel('Volatility Error', color='tab:red')
    ax2.plot(epochs, history['volatility_error'], label='Vol Error', color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')
    
    plt.title(f'Training Metrics for {ticker}')
    fig.tight_layout()
    plt.grid(True, alpha=0.3)
    plt.show()

if execute_train == True:
    # Execute plot
    plot_training_results(history)

In [ ]:
if not execute_train:
    print("Training skipped as per configuration.")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    val_iter = iter(val_loader)
    test_batch = next(val_iter)

    with torch.no_grad():
        samples, observed, target_mask, obs_mask, obs_tp = model.evaluate(test_batch, n_samples=num_samples)

    print("Generated Samples shape:", samples.shape)


Training skipped as per configuration.


AttributeError: 'FinancialDataset' object has no attribute 'close_idx'

In [ ]:
if not execute_train:
    save_path = f"../outputs/{ticker}_{start_date}_{end_date}_{today_str}_csdi_out.pt"

    payload = {
        "samples": samples.detach().cpu(),
        "observed": observed.detach().cpu() if torch.is_tensor(observed) else observed,
        "target_mask": target_mask.detach().cpu(),
        "obs_mask": obs_mask.detach().cpu(),
        "obs_tp": obs_tp.detach().cpu(),
        "n_samples": num_samples,
    }

    torch.save(payload, save_path)
    print(f"Saved to {save_path}")


Saved to ../outputs/AAPL_2010-01-01_2024-12-31_csdi_out.pt


In [ ]:
# # 1. De-standardize: z -> r
#     # samples is (B, n_s, K, L)
# if execute_train == False:
#     generated_returns = samples.cpu().numpy()

#     # 2. Reconstruct Price paths for the 'Close' price (Feature index 3)
#     # We take the first sample in the batch [0] and first generated path [0]
#     feature_idx = 3 # Close price
#     S0 = 100 # Reference starting price
#     generated_path_r = generated_returns[0, 0, feature_idx, :]
#     reconstructed_prices = S0 * np.exp(np.cumsum(generated_path_r))

#     plt.figure(figsize=(10, 5))
#     plt.plot(reconstructed_prices, label=f"Generated {ticker} Path")
#     plt.title("Reconstructed Price Path from CSDI Checkpoint")
#     plt.ylabel("Standardized Price ($)")
#     plt.legend()
#     plt.show()